# Module 12 Instructor Colab Notebook: Markov Chains and Stochastic Processes

## A Mathematical Bridge to Artificial Intelligence

A stochastic process is a random system evolving over time. A Markov model assumes that the current state contains the information needed to predict the next state. This idea supports models of user behavior, queues, reliability, web navigation, and reinforcement learning.

This notebook distinguishes discrete and continuous state spaces, states the Markov property precisely, develops transition matrices and state distributions, and connects a fixed reinforcement-learning policy to an induced Markov chain.

**Recommended pacing:** three class meetings. The absorbing-chain calculations and reinforcement-learning extension may be adjusted to match course time.


> **Course convention: column-stochastic transition matrices**
>
> Throughout this notebook,
>
> $$
> P_{ij}=P(X_{t+1}=i\mid X_t=j).
> $$
>
> Columns represent current states, rows represent next states, every column sums to one, state distributions are column vectors, and
>
> $$
> \mathbf{x}_{t+1}=P\mathbf{x}_t.
> $$
>
> Many books use row-stochastic matrices instead. Either convention works when used consistently; formulas must not be mixed.

## Notebook map

| Section | Main mathematical ideas | Direct AI connection |
|---|---|---|
| 1 | Stochastic processes and state spaces | Representing evolving systems |
| 2 | Markov property | State-based prediction |
| 3 | Transition matrices and multi-step probabilities | Distribution propagation |
| 4 | Stationary and absorbing behavior | Long-run and terminal outcomes |
| 5 | Simulation and empirical frequencies | Sampling trajectories |
| 6 | Reinforcement-learning connection | Policies induce Markov chains |
| Integrated lab | Three-state system | Forecast, simulate, and find steady state |


In [ ]:
# Core setup: compatible with Google Colab and standard Jupyter.
import math
import heapq
from collections import deque
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

np.set_printoptions(precision=5, suppress=True)
plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

try:
    import ipywidgets as widgets
    from ipywidgets import interact, FloatSlider, IntSlider
    WIDGETS_AVAILABLE = True
except Exception:
    WIDGETS_AVAILABLE = False

print("Setup complete.")
print("Interactive widgets available:", WIDGETS_AVAILABLE)


In [ ]:
def reveal_markdown(button_text, solution_markdown):
    if not WIDGETS_AVAILABLE:
        display(Markdown("**Instructor solution**"))
        display(Markdown(solution_markdown))
        return

    button = widgets.Button(description=button_text, button_style="info")
    output = widgets.Output()
    state = {"open": False}

    def toggle(_):
        with output:
            output.clear_output()
            state["open"] = not state["open"]
            if state["open"]:
                display(Markdown(solution_markdown))
                button.description = "Hide instructor solution"
            else:
                button.description = button_text

    button.on_click(toggle)
    display(button, output)


def draw_graph(vertices, edges, positions, directed=False, weights=None,
               title="Graph", highlighted_edges=None, highlighted_vertices=None):
    highlighted_edges = set(highlighted_edges or [])
    highlighted_vertices = set(highlighted_vertices or [])
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.axis("off")
    for index, edge in enumerate(edges):
        u, v = edge
        color = "crimson" if edge in highlighted_edges or (v, u) in highlighted_edges else "#475569"
        arrowstyle = "->" if directed else "-"
        ax.annotate("", xy=positions[v], xytext=positions[u],
                    arrowprops=dict(arrowstyle=arrowstyle, linewidth=2.5,
                                    color=color, shrinkA=18, shrinkB=18))
        if weights is not None:
            midpoint = ((positions[u][0]+positions[v][0])/2,
                        (positions[u][1]+positions[v][1])/2)
            ax.text(midpoint[0], midpoint[1]+0.05, str(weights[index]),
                    ha="center", va="center",
                    bbox=dict(facecolor="white", edgecolor="none", pad=1))
    for vertex in vertices:
        color = "#fecaca" if vertex in highlighted_vertices else "#dbeafe"
        ax.text(positions[vertex][0], positions[vertex][1], str(vertex),
                ha="center", va="center", fontsize=12,
                bbox=dict(boxstyle="circle,pad=0.45", facecolor=color, edgecolor="#1d4ed8"))
    ax.set_title(title)
    plt.show()


## Module learning objectives

By the end of the module, students should be able to:

1. distinguish a random variable from a stochastic process;
2. classify time and state spaces as discrete or continuous;
3. state and interpret the Markov property;
4. construct and validate a column-stochastic transition matrix;
5. propagate state distributions through several steps;
6. identify stationary and absorbing states;
7. simulate sample paths; and
8. explain how a fixed reinforcement-learning policy induces a Markov chain.


---

# Section 1: Stochastic Processes and State Spaces

## 1. From one random variable to a time-indexed family

A **stochastic process** is a collection of random variables indexed by time:

$$
\{X_t:t\in T\}.
$$

The index set $$T$$ is the **time space**, and the set of possible values of $$X_t$$ is the **state space** $$S.$$

A realized sequence

$$
x_0,x_1,x_2,\ldots
$$

is called a **sample path** or trajectory. The stochastic process is the probability model for all possible paths, not one observed path.


## 2. Discrete and continuous time

In **discrete time**, observations occur at separated indices:

$$
t=0,1,2,\ldots.
$$

Examples include model status after each request or a robot location after each action.

In **continuous time**, the process is indexed at every real time in an interval:

$$
t\geq 0.
$$

Examples include continuously monitored temperature or the number of arrivals up to time $$t.$$


## 3. Discrete and continuous state spaces

A **discrete state space** is finite or countable, such as

$$
S=\{\text{idle},\text{busy},\text{failed}\}.
$$

A **continuous state space** contains intervals or regions, such as

$$
S=\mathbb{R}^2
$$

for a physical position.

Time type and state type are independent choices. A process can have discrete time and continuous states, as in daily temperature, or continuous time and discrete states, as in a device switching between on and off.


## 4. State representation matters

A **state** should contain the information used to predict future evolution. If important history is omitted, a process that appears non-Markovian may become Markovian after the state is expanded.

For example, robot position alone may be insufficient if future motion depends on velocity. An expanded state

$$
\text{state}=(\text{position},\text{velocity})
$$

can restore a useful Markov description.

State design is therefore a modeling decision, not merely a label.


> **Common mistakes in Section 1**
>
> - Treating one observed path as the entire stochastic process.
> - Assuming discrete time requires a discrete state space.
> - Confusing an observation with the underlying state.
> - Omitting information needed for future prediction from the state.
> - Assuming every time-dependent random system is Markovian.

## Section 1 terminology

| Term | Meaning |
|---|---|
| Stochastic process | Time-indexed collection of random variables |
| Time space | Set of time indices |
| State space | Set of possible process states |
| Discrete time | Time represented by separated indices |
| Continuous time | Time represented throughout real intervals |
| Sample path | One realized state trajectory |
| State representation | Chosen information used to describe current system condition |


## Section 1 guided worksheet

Classify the time and state spaces for each process:

1. number of service requests in the system at each minute;
2. temperature recorded continuously;
3. robot position in the plane after each action;
4. device status, on or off, at every real time.

For the robot, explain one reason position alone might be an incomplete state.


In [ ]:
reveal_markdown("Open Section 1 solutions", r"""
1. Time is discrete and the count state space is discrete.
2. Time is continuous and temperature is modeled with a continuous state space.
3. Time is discrete while planar position is continuous.
4. Time is continuous while the two-state status space is discrete.

Position may be incomplete if the next position also depends on velocity, orientation, remaining battery, or another hidden quantity not recoverable from position alone.
""")


---

# Section 2: The Markov Property

## 1. Conditional independence from the distant past

A discrete-time stochastic process satisfies the **Markov property** when

$$
\boxed{P(X_{t+1}=j\mid X_t=i,X_{t-1}=i_{t-1},\ldots,X_0=i_0)
=P(X_{t+1}=j\mid X_t=i).}
$$

Once the current state is known, earlier states provide no additional information about the next-state distribution.

The phrase “memoryless” describes the model's conditional structure. It does not mean a state cannot encode history.


## 2. Markov chain

A **Markov chain** is a discrete-time Markov process with a discrete state space. A chain is **time homogeneous** when transition probabilities do not depend on the time index:

$$
P(X_{t+1}=i\mid X_t=j)=P_{ij}
$$

for every $$t.$$

Time homogeneity does not mean states are equally likely over time. It means the transition rule stays fixed.


## 3. Example and non-example

Suppose a system has states normal and overloaded, and tomorrow's state probability depends only on today's state. This is a two-state Markov model.

If tomorrow's overload probability also depends on how many consecutive overloaded days have occurred, then the two-state description is not Markovian. One can expand the state to include duration:

$$
(\text{status},\text{consecutive duration}).
$$

The expanded model may then satisfy the Markov property.


## 4. Higher-order dependence and state augmentation

A second-order model might satisfy

$$
P(X_{t+1}\mid X_t,X_{t-1},\ldots)
=P(X_{t+1}\mid X_t,X_{t-1}).
$$

Define an augmented state

$$
Y_t=(X_t,X_{t-1}).
$$

Then $$Y_t$$ can be first-order Markov. This illustrates a general principle: a Markov assumption is always relative to a chosen state description.


> **Common mistakes in Section 2**
>
> - Saying a Markov system has no history.
> - Assuming the Markov property follows from time homogeneity.
> - Confusing independence of consecutive states with conditional independence given the current state.
> - Declaring a process non-Markovian without considering state augmentation.
> - Treating a modeling assumption as automatically true of observed data.

## Section 2 terminology

| Term | Meaning |
|---|---|
| Markov property | Future conditionally independent of earlier past given current state |
| Markov chain | Discrete-time, discrete-state Markov process |
| Time homogeneous | Transition rule does not change with time index |
| Memoryless description | Next-state law depends on current represented state only |
| State augmentation | Adding information to make current state sufficient |
| Higher-order model | Model depending on several recent states |


## Section 2 guided worksheet

For each situation, decide whether the stated representation is plausibly Markovian:

1. next weather category depends only on today's category;
2. next robot position depends on current position and unrecorded velocity;
3. next word depends on the entire token context stored in the state;
4. machine failure probability depends on current status and unrecorded age.

Suggest a state augmentation for each inadequate representation.


In [ ]:
reveal_markdown("Open Section 2 solutions", r"""
1. This is Markovian under the stated modeling assumption.
2. Position alone is inadequate; add velocity and possibly orientation.
3. If the state truly contains the relevant entire context, the representation can be Markovian at the token-generation level.
4. Status alone is inadequate; include machine age, cumulative use, or a sufficient degradation measure.

These are modeling judgments that should be checked against mechanisms and data.
""")


---

# Section 3: Transition Matrices and Distribution Propagation

## 1. Transition matrix

Under the course convention,

$$
P_{ij}=P(X_{t+1}=i\mid X_t=j).
$$

For each current state $$j,$$ exactly one next state occurs, so

$$
\sum_iP_{ij}=1.
$$

Thus every column sums to one. All entries must also be nonnegative.

The state distribution is

$$
\mathbf{x}_t=
\begin{bmatrix}
P(X_t=1)\\
\vdots\\
P(X_t=n)
\end{bmatrix},
$$

and evolves by

$$
\boxed{\mathbf{x}_{t+1}=P\mathbf{x}_t.}
$$


## 2. Detailed two-state music example

Let states be R and P for two listening categories. Use

$$
P=
\begin{bmatrix}
0.80&0.30\\
0.20&0.70
\end{bmatrix}.
$$

The first column says a current R listener stays in R with probability $$0.80$$ and switches to P with probability $$0.20.$$ The second column says a current P listener switches to R with probability $$0.30$$ and stays in P with probability $$0.70.$$

Let

$$
\mathbf{x}_0=
\begin{bmatrix}
0.21\\
0.79
\end{bmatrix}.
$$

Then

$$
\mathbf{x}_1=P\mathbf{x}_0
=
\begin{bmatrix}
0.80&0.30\\
0.20&0.70
\end{bmatrix}
\begin{bmatrix}
0.21\\
0.79
\end{bmatrix}.
$$

The first component is

$$
0.80(0.21)+0.30(0.79)=0.168+0.237=0.405.
$$

The second is

$$
0.20(0.21)+0.70(0.79)=0.042+0.553=0.595.
$$

Therefore

$$
\boxed{\mathbf{x}_1=
\begin{bmatrix}
0.405\\
0.595
\end{bmatrix}.}
$$


## 3. Multi-step transition probabilities

Apply the transition repeatedly:

$$
\mathbf{x}_2=P\mathbf{x}_1=P(P\mathbf{x}_0)=P^2\mathbf{x}_0.
$$

In general,

$$
\boxed{\mathbf{x}_t=P^t\mathbf{x}_0.}
$$

The entry of $$P^t$$ in row $$i$$ and column $$j$$ is the probability of being in state $$i$$ after $$t$$ steps given initial state $$j.$$


In [ ]:
P_music12 = np.array([[0.80, 0.30], [0.20, 0.70]])
x0_music12 = np.array([0.21, 0.79])
music_distributions12 = [x0_music12]
for _ in range(12):
    music_distributions12.append(P_music12 @ music_distributions12[-1])
music_distributions12 = np.array(music_distributions12)

fig, ax = plt.subplots()
ax.plot(music_distributions12[:, 0], marker="o", label="state R")
ax.plot(music_distributions12[:, 1], marker="o", label="state P")
ax.set_xlabel("step")
ax.set_ylabel("state probability")
ax.set_ylim(0, 1)
ax.set_title("A state distribution evolves through matrix multiplication")
ax.legend()
plt.show()


## 4. Transition graph

A transition graph uses one vertex per state and a directed edge for every positive transition probability. Edge weights are transition probabilities.

Self-loops represent remaining in the same state. Outgoing probabilities from a current state sum to one. Under our matrix convention, outgoing edges correspond to one matrix column.


> **Common mistakes in Section 3**
>
> - Mixing row-stochastic and column-stochastic formulas.
> - Reading the matrix indices backward.
> - Failing to check nonnegativity and column sums.
> - Using elementwise powers instead of matrix powers.
> - Treating a state distribution as a state label.

## Section 3 terminology

| Term | Meaning |
|---|---|
| Transition probability | Conditional probability of one next-state move |
| Transition matrix | Matrix collecting all one-step transitions |
| Column-stochastic | Nonnegative matrix whose columns sum to one |
| State distribution | Vector of probabilities across states |
| Multi-step transition | Probability after repeated transitions |
| Transition graph | Directed weighted graph of possible state moves |
| Self-loop | Transition from a state back to itself |


## Section 3 guided worksheet

Use

$$
P=
\begin{bmatrix}
0.6&0.2\\
0.4&0.8
\end{bmatrix}
$$

and

$$
\mathbf{x}_0=
\begin{bmatrix}
0.75\\
0.25
\end{bmatrix}.
$$

1. Verify the matrix is valid under the course convention.
2. Compute $$\mathbf{x}_1.$$
3. Compute $$\mathbf{x}_2.$$
4. Interpret the first column.


In [ ]:
reveal_markdown("Open Section 3 solutions", r"""
1. Entries are nonnegative and columns sum to

   $$
   0.6+0.4=1
   $$

   and

   $$
   0.2+0.8=1.
   $$

2. Compute

   $$
   \mathbf{x}_1=P\mathbf{x}_0
   =
   \begin{bmatrix}
   0.6(0.75)+0.2(0.25)\\
   0.4(0.75)+0.8(0.25)
   \end{bmatrix}
   =
   \boxed{
   \begin{bmatrix}
   0.50\\
   0.50
   \end{bmatrix}}.
   $$

3. Then

   $$
   \mathbf{x}_2=P\mathbf{x}_1
   =
   \begin{bmatrix}
   0.6(0.5)+0.2(0.5)\\
   0.4(0.5)+0.8(0.5)
   \end{bmatrix}
   =
   \boxed{
   \begin{bmatrix}
   0.4\\
   0.6
   \end{bmatrix}}.
   $$

4. From current state one, next state is one with probability $$0.6$$ and state two with probability $$0.4.$$
""")


---

# Section 4: Stationary, Transient, and Absorbing Behavior

## 1. Stationary distributions

A **stationary distribution** $$\boldsymbol{\pi}$$ satisfies

$$
\boxed{P\boldsymbol{\pi}=\boldsymbol{\pi},}
$$

with nonnegative components summing to one. If the chain begins in this distribution, its distribution remains unchanged.

Stationary does not mean a sampled system remains in one state. Individual paths can continue moving while the population proportions remain stable.


## 2. Solve the music stationary distribution

Let

$$
\boldsymbol{\pi}=
\begin{bmatrix}
r\\
p
\end{bmatrix},
\qquad
r+p=1.
$$

The equation $$P\boldsymbol{\pi}=\boldsymbol{\pi}$$ gives

$$
0.80r+0.30p=r.
$$

Rearrange:

$$
0.30p=0.20r.
$$

Thus

$$
3p=2r
$$

and

$$
r=\frac{3}{2}p.
$$

Use normalization:

$$
\frac{3}{2}p+p=1.
$$

Therefore

$$
\frac{5}{2}p=1,
$$

$$
p=0.4,
$$

and

$$
r=0.6.
$$

Hence

$$
\boxed{\boldsymbol{\pi}=
\begin{bmatrix}
0.6\\
0.4
\end{bmatrix}.}
$$


## 3. Absorbing states

A state is **absorbing** when, once entered, it cannot be left:

$$
P(X_{t+1}=i\mid X_t=i)=1.
$$

Under our convention, the corresponding column has a one in row $$i$$ and zeros elsewhere.

A state is **transient** if the process can leave it and may never return. A chain can have several absorbing outcomes.


## 4. Detailed absorbing example

Let states be search, success, and failure, with

$$
P=
\begin{bmatrix}
0.5&0&0\\
0.3&1&0\\
0.2&0&1
\end{bmatrix}.
$$

Success and failure are absorbing. From search, the system remains searching with probability $$0.5,$$ succeeds with probability $$0.3,$$ and fails with probability $$0.2.$$

Starting in search,

$$
\mathbf{x}_0=
\begin{bmatrix}
1\\
0\\
0
\end{bmatrix},
$$

the first step is

$$
\mathbf{x}_1=
\begin{bmatrix}
0.5\\
0.3\\
0.2
\end{bmatrix}.
$$

The probability of still searching after $$t$$ steps is

$$
(0.5)^t\to 0.
$$

Conditional on eventually leaving search, the relative success and failure probabilities are

$$
\frac{0.3}{0.3+0.2}=0.6
$$

and

$$
\frac{0.2}{0.3+0.2}=0.4.
$$


In [ ]:
P_absorb12 = np.array([[0.5, 0.0, 0.0], [0.3, 1.0, 0.0], [0.2, 0.0, 1.0]])
x_absorb12 = np.array([1.0, 0.0, 0.0])
absorb_history12 = [x_absorb12]
for _ in range(12):
    absorb_history12.append(P_absorb12 @ absorb_history12[-1])
absorb_history12 = np.array(absorb_history12)

fig, ax = plt.subplots()
for index, label in enumerate(["search", "success", "failure"]):
    ax.plot(absorb_history12[:, index], marker="o", label=label)
ax.set_xlabel("step")
ax.set_ylabel("state probability")
ax.set_ylim(0, 1)
ax.set_title("Probability flows from a transient state into absorbing states")
ax.legend()
plt.show()


> **Common mistakes in Section 4**
>
> - Interpreting a stationary distribution as a frozen trajectory.
> - Assuming every finite chain has one unique stationary distribution.
> - Confusing an absorbing state with a high self-loop probability below one.
> - Ignoring multiple absorbing outcomes.
> - Treating transient probability at finite time as eventual absorption probability.

## Section 4 terminology

| Term | Meaning |
|---|---|
| Stationary distribution | Distribution unchanged by the transition matrix |
| Long-run distribution | Limiting distribution when convergence conditions hold |
| Absorbing state | State with probability one of remaining there |
| Transient state | State that can be left and may not be revisited |
| Absorption probability | Probability of eventual entry into an absorbing state |
| Invariant distribution | Another name for a stationary distribution |


## Section 4 guided worksheet

Use

$$
P=
\begin{bmatrix}
0.7&0&0\\
0.2&1&0\\
0.1&0&1
\end{bmatrix}.
$$

1. Identify transient and absorbing states.
2. Starting from state one, find the distribution after one step.
3. Find the probability of remaining in state one after five steps.
4. Find eventual absorption probabilities into states two and three.


In [ ]:
reveal_markdown("Open Section 4 solutions", r"""
1. State one is transient. States two and three are absorbing.

2. The first column gives

   $$
   \boxed{
   \begin{bmatrix}
   0.7\\
   0.2\\
   0.1
   \end{bmatrix}}.
   $$

3. Remaining in state one for five transitions has probability

   $$
   \boxed{(0.7)^5=0.16807}.
   $$

4. When the process leaves state one, relative exit probabilities are

   $$
   \frac{0.2}{0.2+0.1}=\boxed{\frac{2}{3}}
   $$

   and

   $$
   \frac{0.1}{0.2+0.1}=\boxed{\frac{1}{3}}.
   $$
""")


---

# Section 5: Simulation and Empirical State Frequencies

## 1. Distribution evolution versus sample paths

Matrix multiplication computes exact model probabilities. Simulation generates random sample paths. One path can look irregular even when the distribution evolves smoothly.

To simulate under the column convention, if the current state is $$j,$$ sample the next state from column $$j$$:

$$
P[:,j].
$$

Repeating many independent paths produces empirical frequencies that approximate theoretical probabilities.


In [ ]:
def simulate_markov_column(P, initial_state, steps, rng):
    state = initial_state
    path = [state]
    for _ in range(steps):
        state = int(rng.choice(P.shape[0], p=P[:, state]))
        path.append(state)
    return np.array(path)

rng_markov12 = np.random.default_rng(1212)
example_path12 = simulate_markov_column(P_music12, 1, 30, rng_markov12)
print("One sample path using state indices zero and one:")
print(example_path12)


In [ ]:
rng_many12 = np.random.default_rng(3434)
path_count12 = 20000
step_count12 = 8
ending_states12 = np.array([
    simulate_markov_column(P_music12, 1, step_count12, rng_many12)[-1]
    for _ in range(path_count12)
])
empirical_distribution12 = np.bincount(ending_states12, minlength=2)/path_count12
theoretical_distribution12 = np.linalg.matrix_power(P_music12, step_count12) @ np.array([0.0, 1.0])
print("Empirical distribution:", empirical_distribution12)
print("Theoretical distribution:", theoretical_distribution12)


## 2. Monte Carlo error

Empirical frequencies vary because only finitely many paths are simulated. Increasing the number of paths generally reduces sampling variation, but does not correct an incorrect transition model.

Simulation is especially useful when exact state-space calculations are too large, but validation should compare code against small cases with known answers.


> **Common mistakes in Section 5**
>
> - Comparing one path with a probability vector as though they were the same object.
> - Sampling from a row when using the column convention.
> - Expecting empirical frequencies to equal theory exactly.
> - Using more simulations to hide model misspecification.
> - Failing to set or report a random seed for reproducibility.

## Section 5 terminology

| Term | Meaning |
|---|---|
| Simulation | Random generation from a specified model |
| Empirical frequency | Observed fraction across simulated or real cases |
| Theoretical probability | Probability implied by the model |
| Monte Carlo error | Sampling variation caused by finitely many simulations |
| Random seed | Initializer supporting reproducible pseudorandom draws |
| Trajectory ensemble | Collection of simulated sample paths |


## Section 5 guided worksheet

1. Describe how to simulate one transition from current state two under the course convention.
2. Explain why one path cannot equal a distribution.
3. Predict what happens to empirical frequency variability when simulation count quadruples.
4. State one error that more simulation cannot fix.


In [ ]:
reveal_markdown("Open Section 5 solutions", r"""
1. Use the second matrix column as the categorical probability vector and sample one next-state index from it.
2. A path occupies one state at each time. A distribution assigns probabilities across all states and summarizes an ensemble of possible paths.
3. Standard Monte Carlo frequency error typically scales approximately with the reciprocal square root of simulation count. Quadrupling the count roughly halves the typical sampling error.
4. More simulation cannot fix wrong transition probabilities, omitted states, a violated Markov assumption, or incorrect code.
""")


---

# Section 6: Connection to Reinforcement Learning

## 1. Markov decision process

A **Markov decision process**, or MDP, adds actions and rewards to state transitions. Its main components are:

- state space $$S$$;
- action space $$A$$;
- transition model

  $$
  P(s'\mid s,a);
  $$

- reward model; and
- a discount factor for future rewards when applicable.

The Markov property states that, given current state and action, the next-state distribution does not require earlier history.


## 2. Policy

A **policy** specifies how actions are selected. A stochastic policy is

$$
\pi(a\mid s)=P(A_t=a\mid S_t=s).
$$

Once a policy is fixed, the state process has transition probabilities

$$
P_{\pi}(s'\mid s)
=\sum_aP(s'\mid s,a)\pi(a\mid s).
$$

Therefore a fixed policy induces a Markov chain over states.


## 3. Expected immediate reward

If each state has reward $$r_i$$ and the current state distribution is $$\mathbf{x}_t,$$ expected reward is

$$
E[R_t]=\mathbf{r}^T\mathbf{x}_t.
$$

With discount factor $$0\leq\gamma<1,$$ a discounted return is

$$
G_t=R_t+\gamma R_{t+1}+\gamma^2R_{t+2}+\cdots.
$$

Markov transition dynamics determine future state probabilities, while the reward function assigns value to the resulting states or transitions.


## 4. State versus observation in reinforcement learning

An agent may receive an observation that does not reveal the full state. If hidden information affects future transitions, the observation process may not be Markovian from the agent's perspective. This leads to partially observable models.

Memory, belief states, or recurrent representations can summarize history, but whether they are sufficient is a modeling and learning question.


> **Common mistakes in Section 6**
>
> - Confusing a Markov chain with an MDP that includes decisions.
> - Treating a policy as a transition matrix without combining it with action dynamics.
> - Confusing reward with transition probability.
> - Assuming the agent's observation is always the true Markov state.
> - Maximizing immediate reward while ignoring future consequences.

## Section 6 terminology

| Term | Meaning |
|---|---|
| Markov decision process | Markov state model with actions and rewards |
| Action | Decision available to an agent |
| Policy | Rule assigning action probabilities by state |
| Reward | Numerical signal evaluating an outcome or transition |
| Discount factor | Weight reducing later reward contributions |
| Induced Markov chain | State transition chain produced by a fixed policy |
| Partial observability | Agent observations do not reveal the full state |


## Section 6 guided worksheet

At one state, an agent chooses action $$a_1$$ with probability $$0.7$$ and action $$a_2$$ with probability $$0.3.$$ The probability of moving to a target state is $$0.8$$ under $$a_1$$ and $$0.2$$ under $$a_2.$$

1. Compute the target-state transition probability under the policy.
2. Explain why this is a weighted average.
3. State what additional information is needed to compute expected reward.
4. Explain how changing the policy changes the induced Markov chain.


In [ ]:
reveal_markdown("Open Section 6 solutions", r"""
1. Average over actions:

   $$
   P_{\pi}(\text{target}\mid s)
   =(0.8)(0.7)+(0.2)(0.3)
   =0.56+0.06
   =\boxed{0.62}.
   $$

2. The action is random under the policy, so total probability sums each action-specific transition probability weighted by the probability of selecting that action.
3. One needs the reward assigned to the target and other possible outcomes, or a complete reward model for state-action-next-state combinations.
4. Different action probabilities change the weighted transition probabilities and therefore change entries of the induced transition matrix.
""")


---

# Integrated Instructor Lab: A Three-State Markov System

Use states $$A,B,C$$ and transition matrix

$$
P=
\begin{bmatrix}
0.6&0.2&0.1\\
0.3&0.5&0.2\\
0.1&0.3&0.7
\end{bmatrix}.
$$

Every column sums to one. Start in state $$A$$:

$$
\mathbf{x}_0=
\begin{bmatrix}
1\\
0\\
0
\end{bmatrix}.
$$

Then

$$
\mathbf{x}_1=P\mathbf{x}_0
=
\begin{bmatrix}
0.6\\
0.3\\
0.1
\end{bmatrix}.
$$

Next,

$$
\mathbf{x}_2=P\mathbf{x}_1.
$$

Compute components:

$$
x_{2,A}=0.6(0.6)+0.2(0.3)+0.1(0.1)=0.43,
$$

$$
x_{2,B}=0.3(0.6)+0.5(0.3)+0.2(0.1)=0.35,
$$

$$
x_{2,C}=0.1(0.6)+0.3(0.3)+0.7(0.1)=0.22.
$$

Therefore

$$
\boxed{\mathbf{x}_2=
\begin{bmatrix}
0.43\\
0.35\\
0.22
\end{bmatrix}.}
$$


## Stationary distribution

Solve

$$
P\boldsymbol{\pi}=\boldsymbol{\pi}
$$

with components summing to one. The solution is

$$
\boxed{\boldsymbol{\pi}=
\begin{bmatrix}
9/34\\
11/34\\
14/34
\end{bmatrix}
\approx
\begin{bmatrix}
0.264706\\
0.323529\\
0.411765
\end{bmatrix}.}
$$

For example, the first component verifies

$$
0.6\left(\frac{9}{34}\right)
+0.2\left(\frac{11}{34}\right)
+0.1\left(\frac{14}{34}\right)
=\frac{9}{34}.
$$


In [ ]:
P_lab12 = np.array([[0.6, 0.2, 0.1], [0.3, 0.5, 0.2], [0.1, 0.3, 0.7]])
x0_lab12 = np.array([1.0, 0.0, 0.0])
history_lab12 = [x0_lab12]
for _ in range(30):
    history_lab12.append(P_lab12 @ history_lab12[-1])
history_lab12 = np.array(history_lab12)

A_stationary12 = np.vstack([P_lab12-np.eye(3), np.ones(3)])
b_stationary12 = np.concatenate([np.zeros(3), [1.0]])
stationary_lab12 = np.linalg.lstsq(A_stationary12, b_stationary12, rcond=None)[0]

fig, ax = plt.subplots()
for index, label in enumerate(["A", "B", "C"]):
    ax.plot(history_lab12[:, index], marker="o", markersize=3, label=label)
    ax.axhline(stationary_lab12[index], linestyle="--", alpha=0.5)
ax.set_xlabel("step")
ax.set_ylabel("state probability")
ax.set_ylim(0, 1)
ax.set_title("Distribution converges toward the stationary proportions")
ax.legend()
plt.show()

print("Stationary distribution:", stationary_lab12)


## Integrated lab tasks

1. Verify every matrix column sum.
2. Compute $$P^2$$ and interpret its first column.
3. Verify all three stationary equations using fractions.
4. Assign rewards $$2,$$ $$1,$$ and $$-1$$ to states $$A,B,C$$ and compute stationary expected reward.
5. Simulate many paths and compare empirical state frequencies with the theoretical distribution.


In [ ]:
reveal_markdown("Open integrated-lab solution", r"""
The stationary expected reward is

$$
\mathbf{r}^T\boldsymbol{\pi}
=2\left(\frac{9}{34}\right)
+1\left(\frac{11}{34}\right)
-1\left(\frac{14}{34}\right).
$$

Thus

$$
E[R]=\frac{18+11-14}{34}
=\boxed{\frac{15}{34}\approx 0.441176}.
$$

The first column of $$P^2$$ is the two-step state distribution from a definite start in state $$A,$$ namely

$$
\begin{bmatrix}
0.43\\
0.35\\
0.22
\end{bmatrix}.
$$

Simulation frequencies should be close but not identical to the theoretical probabilities because of Monte Carlo variation.
""")


---

# Module 12 Summary

1. A stochastic process is a time-indexed collection of random variables.
2. Time and state spaces can each be discrete or continuous.
3. The Markov property is conditional independence of the future from earlier history given current state.
4. Under the course convention, transition-matrix columns sum to one and distributions update by $$P\mathbf{x}.$$
5. Matrix powers give multi-step transitions.
6. Stationary distributions remain unchanged, while absorbing states cannot be left.
7. Simulation produces sample paths whose ensemble frequencies approximate theoretical probabilities.
8. A fixed reinforcement-learning policy induces a Markov chain over states.

## Exit ticket

1. Why is the Markov property relative to a chosen state representation?
2. What does one transition-matrix column represent in this notebook?
3. Why can a stationary chain still have moving sample paths?
4. What distinguishes an absorbing state from a large self-loop probability?
5. How does a policy convert an MDP into an induced Markov chain?
